In [1]:


from sklearn.metrics import *
import warnings
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from files.functions import *
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

warnings.filterwarnings('ignore')

In [2]:
COIN = 'BTC'
data = pd.read_csv(fullDataPath(COIN))
data

,time,low,high,open,close,volume,change,pct_change,SMA_20,SMA_50,...,EMA_26,MACD,MACD_Signal,MACD_Hist,BB_Middle,BB_STD,BB_Upper,BB_Lower,Volume_MA_20,OBV
0,2024-06-10,69141.51,70187.53,69641.08,69497.73,6088.403632,-143.35,-0.205841,64462.8325,63358.5884,...,64556.456847,1795.539506,1344.659895,450.879611,64462.8325,2772.307164,70007.446829,58918.218171,10009.565107,-177752.513399
1,2024-06-11,66018.69,69547.82,69495.97,67316.53,17354.689155,-2179.44,-3.136067,64121.3590,63292.0274,...,64161.154994,1618.889786,1231.939992,386.949794,64121.3590,2529.471214,69180.301429,59062.416571,9918.306387,-183840.917031
2,2024-06-12,66883.32,70032.00,67316.52,68248.60,17140.011394,932.08,1.384623,63897.0390,63237.8892,...,63908.724994,1591.958837,1135.202544,456.756294,63897.0390,2428.103367,68753.245733,59040.832267,9666.907278,-166486.227877
3,2024-06-13,66200.00,68474.49,68248.59,66738.85,11916.132111,-1509.74,-2.212119,63586.6200,63178.6808,...,63561.534993,1439.527716,1021.013470,418.514246,63586.6200,2231.385312,68049.390623,59123.849377,9344.338774,-183626.239271
4,2024-06-14,65005.00,67322.72,66746.46,66004.39,11520.157919,-742.07,-1.111774,63256.9280,63072.2650,...,63307.349793,1377.751591,916.384909,461.366683,63256.9280,2228.248272,67713.424544,58800.431456,9351.868007,-195542.371382
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2025-06-05,100345.73,105999.68,104753.37,101570.20,10615.733871,-3183.17,-3.038728,75484.8190,67143.1058,...,77454.851181,10591.718400,6471.786251,4119.932149,75484.8190,17724.731021,110934.281043,40035.356957,8960.382431,-199892.255134
361,2025-06-06,101132.91,105439.01,101570.20,104397.99,9674.999986,2827.79,2.784074,73495.7945,66427.6102,...,75525.623275,10062.104411,5441.803214,4620.301198,73495.7945,16854.080008,107203.954516,39787.634484,9189.674862,-189276.521263
362,2025-06-07,103969.70,106000.00,104398.00,105619.02,4141.101243,1221.02,1.169582,71316.7290,65698.1320,...,73215.833937,8951.846056,4286.727914,4665.118142,71316.7290,15403.328372,102123.385745,40510.072255,9275.564780,-179601.521277
363,2025-06-08,105028.30,106548.90,105619.02,105784.40,1998.610110,165.38,0.156582,69116.5475,64943.7398,...,70623.579052,7280.220940,3120.448379,4159.772561,69116.5475,13235.997192,95588.541885,42644.553115,9595.027667,-175460.420034


clean up

In [45]:

data['time'] = pd.to_datetime(data['time'], errors='coerce')
data = data.dropna(subset=['time'])  # optional: drop rows where parsing failed
data['date'] = data['time'].dt.date

# 3.4) Coerce any non-numeric sentiment scores to NaN, then drop those rows:
data['score'] = pd.to_numeric(data['score'], errors='coerce')
data = data.dropna(subset=['score'])

# 3.5) Compute daily average sentiment and tweet count for each date:
daily_sent = (
    data
    .groupby('date')
    .agg(
        avg_sentiment=('score', 'mean'),
        tweet_count=('score', 'size')
    )
    .reset_index()
)

daily_sent.head()

,date,avg_sentiment,tweet_count
0,2024-05-08,0.0,1
1,2024-05-09,0.0,1
2,2024-05-10,0.0,1
3,2024-05-11,0.0,1
4,2024-05-12,0.0,1


In [46]:
price_df = data.copy()
price_df['date'] = pd.to_datetime(data['date'], errors='coerce')
price_df = price_df.dropna(subset=['date'])  # drop rows where parsing failed
price_df['date'] = price_df['date'].dt.date
price_df = price_df.set_index('date').sort_index()

In [47]:
# 5.1) Convert daily_sent to index='date', then join with price_df:
daily_sent = daily_sent.set_index('date').sort_index()

merged = price_df.join(daily_sent, how='inner')
# 'inner' means only dates present in BOTH price_df and daily_sent are kept.

# 5.2) If any days appear in price_df but lack sentiment, forward-fill or set zeros:
merged['avg_sentiment'] = merged['avg_sentiment'].fillna(method='ffill')
merged['tweet_count']    = merged['tweet_count'].fillna(0)

# 5.3) Drop any rows missing price (just in case):
merged = merged.dropna(subset=['close'])

merged.head()

,time,low,high,open,close,volume,change,pct_change,SMA_20,SMA_50,...,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,title,link,text,sentiment,score,avg_sentiment,tweet_count
date,,,,,,,,,,,,,,,,,,,,,
2024-05-08,2024-05-08,60851.04,63013.05,62315.75,61169.53,7486.425968,-1146.22,-1.839374,65890.2210,66417.9682,...,0.0,0.0,0.0,-,-,-,-,0.0,0.0,1
2024-05-09,2024-05-09,60601.60,63424.14,61169.53,63073.57,8360.055382,1904.04,3.112726,66247.8435,66426.8854,...,0.0,0.0,0.0,-,-,-,-,0.0,0.0,1
2024-05-10,2024-05-10,60150.00,63470.00,63073.55,60787.47,11511.129910,-2286.08,-3.624467,66472.6375,66371.6810,...,0.0,0.0,0.0,-,-,-,-,0.0,0.0,1
2024-05-11,2024-05-11,60450.13,61482.00,60787.99,60814.63,2338.068108,26.64,0.043824,66850.1930,66373.6450,...,0.0,0.0,0.0,-,-,-,-,0.0,0.0,1
2024-05-12,2024-05-12,60576.05,61843.45,60814.64,61453.02,2694.975779,638.38,1.049714,67183.0820,66410.7176,...,0.0,0.0,0.0,-,-,-,-,0.0,0.0,1


In [48]:
def make_lag_features(df, col, lags=[1, 2, 3, 7, 14]):
    for lag in lags:
        df[f'{col}_lag_{lag}'] = df[col].shift(lag)
    return df

def make_rolling_features(df, col, windows=[3, 7, 14]):
    for w in windows:
        df[f'{col}_roll_mean_{w}'] = df[col].rolling(w).mean()
        df[f'{col}_roll_std_{w}']  = df[col].rolling(w).std()
    return df

# 6.1) Copy merged so we don’t overwrite
df_feats = merged.copy()

# 6.2) Create lag/rolling for “price”
df_feats = make_lag_features(df_feats, 'price', lags=[1,2,3,7,14])
df_feats = make_rolling_features(df_feats, 'price', windows=[3,7,14])

# 6.3) Create lag/rolling for “avg_sentiment”
df_feats = make_lag_features(df_feats, 'avg_sentiment', lags=[1,2,3,7])
df_feats = make_rolling_features(df_feats, 'avg_sentiment', windows=[3,7])

# 6.4) (Optional) Lag/rolling for “tweet_count”
df_feats = make_lag_features(df_feats, 'tweet_count', lags=[1, 2, 3, 7])
df_feats = make_rolling_features(df_feats, 'tweet_count', windows=[3, 7])

# 6.5) Drop any rows with NaNs generated by shifting/rolling
df_feats = df_feats.dropna().copy()

# 6.6) Create the “next‐day price” target: shift price by –1
df_feats['target_price'] = df_feats['price'].shift(-1)
df_feats = df_feats.dropna(subset=['target_price']).copy()

# 6.7) Define feature matrix X and label y
feature_cols = [c for c in df_feats.columns if c not in ['price', 'target_price']]
X = df_feats[feature_cols]
y = df_feats['target_price']

print("X shape:", X.shape)
print("y shape:", y.shape)

KeyError: 'price'

In [ ]:
# 7.1) Split 80% train / 20% test (chronologically):
total = len(df_feats)
test_size = max(int(total * 0.2), 1)
train_size = total - test_size

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Train rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")

# 7.2) Build a pipeline: StandardScaler + XGBoost
def build_model_pipeline():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('model', XGBRegressor(
            objective='reg:squarederror',
            tree_method='hist',
            random_state=42
        ))
    ])

pipe = build_model_pipeline()

# 7.3) Hyperparameter distributions for RandomizedSearchCV
param_dist = {
    'model__n_estimators':    [50, 100, 200, 400],
    'model__max_depth':       [3, 5, 7, 9],
    'model__learning_rate':   [0.01, 0.05, 0.1, 0.2],
    'model__subsample':       [0.6, 0.8, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
}

tscv = TimeSeriesSplit(n_splits=5)  # 5-fold time-series CV

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_dist,
    n_iter=30,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 7.4) Fit on the training data only
search.fit(X_train, y_train)

print("Best CV RMSE (train):", -search.best_score_)

# 7.5) Get the best model and make predictions
best_model = search.best_estimator_

# Make predictions on test set
y_pred = best_model.predict(X_test)

# Calculate test RMSE
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {test_rmse}")

# 8) Create normalized predictions dataframe
# Get test dates for indexing
test_dates = df_feats.index[train_size:]

# Normalize predictions (scale to 0-1 range based on historical data)
price_min = df_feats['price'].min()
price_max = df_feats['price'].max()
normalized_predictions = (y_pred - price_min) / (price_max - price_min)

# Create predictions dataframe
predictions_df = pd.DataFrame({
    'predictions': normalized_predictions
}, index=test_dates)

# 9) Save predictions to file
import os
os.makedirs(f'../predictions/{COIN}', exist_ok=True)
predictions_df.to_csv(f'../predictions/{COIN}/chat_sentiment_predictions.csv')

print(f"Predictions saved to ../predictions/{COIN}/chat_sentiment_predictions.csv")
print(f"Predictions shape: {predictions_df.shape}")
print("Sample predictions:")
print(predictions_df.head())
